In [4]:
import json
import ssl
import certifi
from urllib.request import Request, urlopen

TEAM_ID = "TEAM_11"
API_KEY = "oc_PLiAzCmqLhyRX2uuS8Xuio0D-uGIotzR"
API_URL = "https://bftrxasgtunepckchcoz.supabase.co/functions/v1/oracle"

ssl_context = ssl.create_default_context(cafile=certifi.where())

def api(payload):
    req = Request(
        API_URL,
        data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json", "X-API-Key": API_KEY},
    )
    with urlopen(req, timeout=30, context=ssl_context) as r:
        return json.load(r)

spec = api({"action": "spec", "team_id": TEAM_ID})
SPACE = spec["hyperparameters"]

print("Model:", spec["model"])
for name, values in SPACE.items():
    print(name, ":", values)

Model: RandomForestRegressor
bootstrap : [True, False]
max_depth : [3, 4, 5, 7, 10, None]
max_features : [0.4, 0.6, 0.8, 1]
n_estimators : [3, 5, 8, 12]
min_samples_leaf : [1, 2, 4, 8]
min_samples_split : [2, 5, 10, 20]


In [5]:
def oracle_query(params):
    result = api({"action": "query", "team_id": TEAM_ID, "params": params})
    return float(result["loss"])

params = {
    "bootstrap": True,
    "max_depth": 5,
    "max_features": 0.8,
    "n_estimators": 8,
    "min_samples_leaf": 2,
    "min_samples_split": 5,
}

loss = oracle_query(params)
print("Loss:", loss)

Loss: 5.36758474552825


In [6]:
params_b = {
    "bootstrap": True,
    "max_depth": 7,
    "max_features": 0.8,
    "n_estimators": 8,
    "min_samples_leaf": 2,
    "min_samples_split": 5,
}
loss_b = oracle_query(params_b)
print("max_depth=7 -> Loss:", loss_b)

max_depth=7 -> Loss: 5.23034158454991


In [7]:
params_c = {
    "bootstrap": True,
    "max_depth": 10,
    "max_features": 0.8,
    "n_estimators": 8,
    "min_samples_leaf": 2,
    "min_samples_split": 5,
}
loss_c = oracle_query(params_c)
print("max_depth=10 -> Loss:", loss_c)

max_depth=10 -> Loss: 5.2626853106236


In [8]:
params_a = {
    "bootstrap": True,
    "max_depth": 7,
    "max_features": 0.8,
    "n_estimators": 5,
    "min_samples_leaf": 2,
    "min_samples_split": 5,
}
loss_a = oracle_query(params_a)
print("n_estimators=5 -> Loss:", loss_a)

params_b = {
    "bootstrap": True,
    "max_depth": 7,
    "max_features": 0.8,
    "n_estimators": 12,
    "min_samples_leaf": 2,
    "min_samples_split": 5,
}
loss_b = oracle_query(params_b)
print("n_estimators=12 -> Loss:", loss_b)

n_estimators=5 -> Loss: 5.62218074481564
n_estimators=12 -> Loss: 4.77658070175863


In [9]:
params_a = {
    "bootstrap": True,
    "max_depth": 7,
    "max_features": 0.6,
    "n_estimators": 12,
    "min_samples_leaf": 2,
    "min_samples_split": 5,
}
loss_a = oracle_query(params_a)
print("max_features=0.6 -> Loss:", loss_a)

params_b = {
    "bootstrap": True,
    "max_depth": 7,
    "max_features": 1,
    "n_estimators": 12,
    "min_samples_leaf": 2,
    "min_samples_split": 5,
}
loss_b = oracle_query(params_b)
print("max_features=1 -> Loss:", loss_b)

max_features=0.6 -> Loss: 4.26095923057704
max_features=1 -> Loss: 4.49605745766431


In [10]:
params_c = {
    "bootstrap": True,
    "max_depth": 7,
    "max_features": 0.4,
    "n_estimators": 12,
    "min_samples_leaf": 2,
    "min_samples_split": 5,
}
loss_c = oracle_query(params_c)
print("max_features=0.4 -> Loss:", loss_c)

max_features=0.4 -> Loss: 5.68955116257175


In [11]:
# --- Oracle wrapper with caching ---
cache = {}
call_log = []

def oracle_query(params):
    result = api({"action": "query", "team_id": TEAM_ID, "params": params})
    return float(result["loss"])

def cached_query(params):
    key = tuple(sorted(params.items(), key=lambda kv: kv[0]))
    if key in cache:
        return cache[key]
    loss = oracle_query(params)
    cache[key] = loss
    call_log.append((dict(params), loss))
    return loss

# --- Pre-seed cache with calls already made manually, so the loop won't repeat them ---
manual_results = [
    ({"bootstrap": True, "max_depth": 5, "max_features": 0.8, "n_estimators": 8,  "min_samples_leaf": 2, "min_samples_split": 5}, 5.36758474552825),
    ({"bootstrap": True, "max_depth": 7, "max_features": 0.8, "n_estimators": 8,  "min_samples_leaf": 2, "min_samples_split": 5}, 5.23034158454991),
    ({"bootstrap": True, "max_depth": 10,"max_features": 0.8, "n_estimators": 8,  "min_samples_leaf": 2, "min_samples_split": 5}, 5.2626853106236),
    ({"bootstrap": True, "max_depth": 7, "max_features": 0.8, "n_estimators": 5,  "min_samples_leaf": 2, "min_samples_split": 5}, 5.62218074481564),
    ({"bootstrap": True, "max_depth": 7, "max_features": 0.8, "n_estimators": 12, "min_samples_leaf": 2, "min_samples_split": 5}, 4.77658070175863),
    ({"bootstrap": True, "max_depth": 7, "max_features": 0.6, "n_estimators": 12, "min_samples_leaf": 2, "min_samples_split": 5}, 4.26095923057704),
    ({"bootstrap": True, "max_depth": 7, "max_features": 1,   "n_estimators": 12, "min_samples_leaf": 2, "min_samples_split": 5}, 4.49605745766431),
    ({"bootstrap": True, "max_depth": 7, "max_features": 0.4, "n_estimators": 12, "min_samples_leaf": 2, "min_samples_split": 5}, 5.68955116257175),
]
for p, l in manual_results:
    key = tuple(sorted(p.items(), key=lambda kv: kv[0]))
    cache[key] = l
    call_log.append((dict(p), l))

# --- Automated directional-probe tuner for one parameter ---
def tune_parameter(name, values, current, current_loss, query_fn):
    """
    Probes outward from the current value's position in `values`.
    Tries both neighbors first to find the improving direction, then keeps
    moving that way, stopping as soon as the loss stops improving.
    Works for both ordered numeric/None lists and short unordered lists (e.g. booleans).
    """
    idx = values.index(current[name])
    best_val, best_loss = current[name], current_loss

    # If only 2 values (e.g. bootstrap), just try the other one directly.
    if len(values) <= 2:
        for val in values:
            if val == current[name]:
                continue
            trial = dict(current)
            trial[name] = val
            loss = query_fn(trial)
            if loss < best_loss:
                best_val, best_loss = val, loss
        return best_val, best_loss

    # Try left neighbor and right neighbor first.
    candidates = []
    if idx - 1 >= 0:
        candidates.append(idx - 1)
    if idx + 1 < len(values):
        candidates.append(idx + 1)

    direction = None
    for cand_idx in candidates:
        trial = dict(current)
        trial[name] = values[cand_idx]
        loss = query_fn(trial)
        if loss < best_loss:
            best_val, best_loss = values[cand_idx], loss
            direction = 1 if cand_idx > idx else -1

    # Keep moving in the improving direction until it stops improving.
    if direction is not None:
        step_idx = values.index(best_val) + direction
        while 0 <= step_idx < len(values):
            trial = dict(current)
            trial[name] = values[step_idx]
            loss = query_fn(trial)
            if loss < best_loss:
                best_val, best_loss = values[step_idx], loss
                step_idx += direction
            else:
                break

    return best_val, best_loss

# --- Run the automated loop over all remaining parameters ---
current = {
    "bootstrap": True,
    "max_depth": 7,
    "max_features": 0.6,
    "n_estimators": 12,
    "min_samples_leaf": 2,
    "min_samples_split": 5,
}
current_loss = cached_query(current)  # pulled from cache, no extra call

PARAM_ORDER = ["min_samples_split", "min_samples_leaf", "bootstrap"]

for name in PARAM_ORDER:
    best_val, best_loss = tune_parameter(name, SPACE[name], current, current_loss, cached_query)
    if best_val != current[name]:
        current[name] = best_val
        current_loss = best_loss
    print(f"Tuned {name}: best value={best_val}, loss={best_loss:.6f}, total calls so far={len(cache)}")

print()
print("Best hyperparameters:", current)
print("Best loss:", current_loss)
print("Total unique Oracle calls used:", len(cache))

Tuned min_samples_split: best value=5, loss=4.260959, total calls so far=10
Tuned min_samples_leaf: best value=2, loss=4.260959, total calls so far=12
Tuned bootstrap: best value=True, loss=4.260959, total calls so far=13

Best hyperparameters: {'bootstrap': True, 'max_depth': 7, 'max_features': 0.6, 'n_estimators': 12, 'min_samples_leaf': 2, 'min_samples_split': 5}
Best loss: 4.26095923057704
Total unique Oracle calls used: 13


In [12]:
# --- Interaction check 1: max_depth x min_samples_leaf ---
depth_candidates = [5, 7, 10]          # around our current best (7)
leaf_candidates = [1, 2, 4]            # around our current best (2)

best_interaction_1 = dict(current)
best_interaction_1_loss = current_loss

print("--- max_depth x min_samples_leaf grid ---")
for d in depth_candidates:
    for l in leaf_candidates:
        trial = dict(current)
        trial["max_depth"] = d
        trial["min_samples_leaf"] = l
        loss = cached_query(trial)
        print(f"max_depth={d}, min_samples_leaf={l} -> Loss: {loss:.6f}")
        if loss < best_interaction_1_loss:
            best_interaction_1_loss = loss
            best_interaction_1 = dict(trial)

print(f"\nBest from grid 1: {best_interaction_1}, loss={best_interaction_1_loss:.6f}")
print(f"Total calls so far: {len(cache)}")

--- max_depth x min_samples_leaf grid ---
max_depth=5, min_samples_leaf=1 -> Loss: 5.603255
max_depth=5, min_samples_leaf=2 -> Loss: 5.368427
max_depth=5, min_samples_leaf=4 -> Loss: 4.944669
max_depth=7, min_samples_leaf=1 -> Loss: 5.060115
max_depth=7, min_samples_leaf=2 -> Loss: 4.260959
max_depth=7, min_samples_leaf=4 -> Loss: 4.714255
max_depth=10, min_samples_leaf=1 -> Loss: 5.096930
max_depth=10, min_samples_leaf=2 -> Loss: 4.675997
max_depth=10, min_samples_leaf=4 -> Loss: 4.879098

Best from grid 1: {'bootstrap': True, 'max_depth': 7, 'max_features': 0.6, 'n_estimators': 12, 'min_samples_leaf': 2, 'min_samples_split': 5}, loss=4.260959
Total calls so far: 19


In [13]:
# --- Interaction check 2: min_samples_split x min_samples_leaf ---
split_candidates = [2, 5, 10]          # around our current best (5)
leaf_candidates = [1, 2, 4]            # around our current best (2)

best_interaction_2 = dict(current)
best_interaction_2_loss = current_loss

print("--- min_samples_split x min_samples_leaf grid ---")
for s in split_candidates:
    for l in leaf_candidates:
        trial = dict(current)
        trial["min_samples_split"] = s
        trial["min_samples_leaf"] = l
        loss = cached_query(trial)
        print(f"min_samples_split={s}, min_samples_leaf={l} -> Loss: {loss:.6f}")
        if loss < best_interaction_2_loss:
            best_interaction_2_loss = loss
            best_interaction_2 = dict(trial)

print(f"\nBest from grid 2: {best_interaction_2}, loss={best_interaction_2_loss:.6f}")
print(f"Total calls so far: {len(cache)}")

--- min_samples_split x min_samples_leaf grid ---
min_samples_split=2, min_samples_leaf=1 -> Loss: 4.758590
min_samples_split=2, min_samples_leaf=2 -> Loss: 4.284823
min_samples_split=2, min_samples_leaf=4 -> Loss: 4.714255
min_samples_split=5, min_samples_leaf=1 -> Loss: 5.060115
min_samples_split=5, min_samples_leaf=2 -> Loss: 4.260959
min_samples_split=5, min_samples_leaf=4 -> Loss: 4.714255
min_samples_split=10, min_samples_leaf=1 -> Loss: 4.723872
min_samples_split=10, min_samples_leaf=2 -> Loss: 5.027135
min_samples_split=10, min_samples_leaf=4 -> Loss: 5.178728

Best from grid 2: {'bootstrap': True, 'max_depth': 7, 'max_features': 0.6, 'n_estimators': 12, 'min_samples_leaf': 2, 'min_samples_split': 5}, loss=4.260959
Total calls so far: 23


In [14]:
print("Total unique Oracle calls used:", len(cache))

Total unique Oracle calls used: 23


In [16]:
def greedy_coordinate_descent(param_names, space, query_fn, start, max_sweeps=2):
    current = dict(start)
    current_loss = query_fn(current)
    for sweep in range(max_sweeps):
        improved = False
        for name in param_names:
            best_val = current[name]
            best_val_loss = current_loss
            for val in space[name]:
                if val == current[name]:
                    continue
                trial = dict(current)
                trial[name] = val
                loss = query_fn(trial)
                if loss < best_val_loss:
                    best_val_loss = loss
                    best_val = val
            if best_val != current[name]:
                current[name] = best_val
                current_loss = best_val_loss
                improved = True
        if not improved:
            break
    return current, current_loss

# --- Two diverse starting points (dropped the middle one to control budget) ---
starting_points = [
    {"bootstrap": False, "max_depth": 3,    "max_features": 0.4, "n_estimators": 3, "min_samples_leaf": 8, "min_samples_split": 20},
    {"bootstrap": True,  "max_depth": None, "max_features": 1,   "n_estimators": 3, "min_samples_leaf": 1, "min_samples_split": 2},
]

PARAM_NAMES = list(SPACE.keys())
restart_results = []

for i, start in enumerate(starting_points):
    result_params, result_loss = greedy_coordinate_descent(PARAM_NAMES, SPACE, cached_query, start, max_sweeps=2)
    restart_results.append((result_params, result_loss))
    print(f"Restart {i+1} (start={start}):")
    print(f"  -> converged to {result_params}, loss={result_loss:.6f}")
    print(f"  -> total calls so far: {len(cache)}")
    print()

print("Current best (from earlier):", current, "loss:", current_loss)
best_of_all = min(restart_results + [(current, current_loss)], key=lambda x: x[1])
print("\nOverall best across everything:", best_of_all[0], "loss:", best_of_all[1])
print("FINAL total unique Oracle calls used:", len(cache))

Restart 1 (start={'bootstrap': False, 'max_depth': 3, 'max_features': 0.4, 'n_estimators': 3, 'min_samples_leaf': 8, 'min_samples_split': 20}):
  -> converged to {'bootstrap': False, 'max_depth': 7, 'max_features': 0.6, 'n_estimators': 12, 'min_samples_leaf': 4, 'min_samples_split': 2}, loss=4.359427
  -> total calls so far: 60

Restart 2 (start={'bootstrap': True, 'max_depth': None, 'max_features': 1, 'n_estimators': 3, 'min_samples_leaf': 1, 'min_samples_split': 2}):
  -> converged to {'bootstrap': True, 'max_depth': None, 'max_features': 1, 'n_estimators': 12, 'min_samples_leaf': 4, 'min_samples_split': 2}, loss=4.295256
  -> total calls so far: 97

Current best (from earlier): {'bootstrap': True, 'max_depth': 7, 'max_features': 0.6, 'n_estimators': 12, 'min_samples_leaf': 2, 'min_samples_split': 5} loss: 4.26095923057704

Overall best across everything: {'bootstrap': True, 'max_depth': 7, 'max_features': 0.6, 'n_estimators': 12, 'min_samples_leaf': 2, 'min_samples_split': 5} loss: 

In [17]:
import itertools
import random

# Build the full space of all possible combinations
all_combos = list(itertools.product(
    SPACE["bootstrap"],
    SPACE["max_depth"],
    SPACE["max_features"],
    SPACE["n_estimators"],
    SPACE["min_samples_leaf"],
    SPACE["min_samples_split"],
))
PARAM_NAMES_ORDERED = ["bootstrap", "max_depth", "max_features", "n_estimators", "min_samples_leaf", "min_samples_split"]

def combo_to_params(combo):
    return dict(zip(PARAM_NAMES_ORDERED, combo))

# Figure out which combos are already cached (as exact key tuples)
already_tried_keys = set(cache.keys())

unvisited_combos = []
for combo in all_combos:
    params = combo_to_params(combo)
    key = tuple(sorted(params.items(), key=lambda kv: kv[0]))
    if key not in already_tried_keys:
        unvisited_combos.append(params)

print(f"Total combinations: {len(all_combos)}")
print(f"Already tried: {len(all_combos) - len(unvisited_combos)}")
print(f"Unvisited remaining: {len(unvisited_combos)}")

# Randomly sample N new configs to test, without repeating anything
N_NEW_SAMPLES = 300   # adjust as desired; total budget allows up to ~824 more
random.seed(42)
sampled_configs = random.sample(unvisited_combos, min(N_NEW_SAMPLES, len(unvisited_combos)))

best_random = dict(current)
best_random_loss = current_loss

for i, params in enumerate(sampled_configs, 1):
    loss = cached_query(params)
    if loss < best_random_loss:
        best_random_loss = loss
        best_random = dict(params)
        print(f"[{i}] NEW BEST: {params} -> loss={loss:.6f}")
    if i % 50 == 0:
        print(f"  ...progress: {i}/{len(sampled_configs)} sampled, total calls so far={len(cache)}, current best={best_random_loss:.6f}")

print()
print("Best from random search:", best_random, "loss:", best_random_loss)
print("Overall best (random search vs. greedy):", min([(best_random, best_random_loss), (current, current_loss)], key=lambda x: x[1]))
print("FINAL total unique Oracle calls used:", len(cache))

Total combinations: 3072
Already tried: 97
Unvisited remaining: 2975
  ...progress: 50/300 sampled, total calls so far=147, current best=4.260959
  ...progress: 100/300 sampled, total calls so far=197, current best=4.260959
  ...progress: 150/300 sampled, total calls so far=247, current best=4.260959
  ...progress: 200/300 sampled, total calls so far=297, current best=4.260959
  ...progress: 250/300 sampled, total calls so far=347, current best=4.260959
  ...progress: 300/300 sampled, total calls so far=397, current best=4.260959

Best from random search: {'bootstrap': True, 'max_depth': 7, 'max_features': 0.6, 'n_estimators': 12, 'min_samples_leaf': 2, 'min_samples_split': 5} loss: 4.26095923057704
Overall best (random search vs. greedy): ({'bootstrap': True, 'max_depth': 7, 'max_features': 0.6, 'n_estimators': 12, 'min_samples_leaf': 2, 'min_samples_split': 5}, 4.26095923057704)
FINAL total unique Oracle calls used: 397


In [18]:
import math
import random

# --- Encode a params dict into a normalized numeric feature vector ---
FEATURE_ORDER = ["bootstrap", "max_depth", "max_features", "n_estimators", "min_samples_leaf", "min_samples_split"]
FEATURE_MINS  = [0,    3,  0.4, 3,  1, 2]
FEATURE_MAXS  = [1,   15,  1.0, 12, 8, 20]   # max_depth=None encoded as sentinel 15

def encode(params):
    b   = 1.0 if params["bootstrap"] else 0.0
    md  = params["max_depth"]
    md  = 15 if md is None else md
    raw = [b, md, params["max_features"], params["n_estimators"], params["min_samples_leaf"], params["min_samples_split"]]
    return [(v - lo) / (hi - lo) for v, lo, hi in zip(raw, FEATURE_MINS, FEATURE_MAXS)]

# --- Squared exponential kernel ---
def kernel(x1, x2, length_scale=0.35, sigma_f=1.0):
    sq_dist = sum((a - b) ** 2 for a, b in zip(x1, x2))
    return sigma_f ** 2 * math.exp(-0.5 * sq_dist / (length_scale ** 2))

# --- Pure-Python Cholesky decomposition and triangular solves ---
def cholesky(A):
    n = len(A)
    L = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(i + 1):
            s = sum(L[i][k] * L[j][k] for k in range(j))
            if i == j:
                L[i][j] = math.sqrt(max(A[i][i] - s, 1e-10))
            else:
                L[i][j] = (A[i][j] - s) / L[j][j]
    return L

def forward_sub(L, b):
    n = len(b)
    y = [0.0] * n
    for i in range(n):
        s = sum(L[i][k] * y[k] for k in range(i))
        y[i] = (b[i] - s) / L[i][i]
    return y

def backward_sub(L, y):
    n = len(y)
    x = [0.0] * n
    for i in reversed(range(n)):
        s = sum(L[k][i] * x[k] for k in range(i + 1, n))
        x[i] = (y[i] - s) / L[i][i]
    return x

# --- GP posterior mean & variance at test points, given training data ---
def gp_predict(X_train, y_train, X_test, noise=1e-4):
    n = len(X_train)
    K = [[kernel(X_train[i], X_train[j]) + (noise if i == j else 0.0) for j in range(n)] for i in range(n)]
    L = cholesky(K)
    alpha = backward_sub(L, forward_sub(L, y_train))

    means, stds = [], []
    for xt in X_test:
        k_star = [kernel(xt, xtr) for xtr in X_train]
        mu = sum(a * k for a, k in zip(alpha, k_star))
        v = forward_sub(L, k_star)
        var = max(kernel(xt, xt) - sum(vi ** 2 for vi in v), 1e-8)
        means.append(mu)
        stds.append(math.sqrt(var))
    return means, stds

# --- Build training set: prioritize best-known points + random fill (cap for speed) ---
sorted_cache = sorted(cache.items(), key=lambda kv: kv[1])
TRAIN_CAP = 100
top_points = sorted_cache[:30]
remaining = sorted_cache[30:]
random.seed(1)
fill = random.sample(remaining, min(TRAIN_CAP - len(top_points), len(remaining)))
train_items = top_points + fill

X_train = [encode(dict(k)) for k, _ in train_items]
y_train = [loss for _, loss in train_items]

# --- Bayesian optimization loop ---
N_BO_ITERS = 12
KAPPA = 2.0  # exploration weight (higher = more exploration)

already_tried_keys = set(cache.keys())
all_combos_full = list(itertools.product(
    SPACE["bootstrap"], SPACE["max_depth"], SPACE["max_features"],
    SPACE["n_estimators"], SPACE["min_samples_leaf"], SPACE["min_samples_split"],
))

bo_best_params, bo_best_loss = dict(current), current_loss

for it in range(N_BO_ITERS):
    unvisited = [combo_to_params(c) for c in all_combos_full
                 if tuple(sorted(combo_to_params(c).items(), key=lambda kv: kv[0])) not in already_tried_keys]
    candidates = random.sample(unvisited, min(150, len(unvisited)))
    X_cand = [encode(p) for p in candidates]

    means, stds = gp_predict(X_train, y_train, X_cand)
    # UCB for minimization: lower score = more promising (low predicted loss, high uncertainty bonus)
    scores = [m - KAPPA * s for m, s in zip(means, stds)]
    best_idx = min(range(len(candidates)), key=lambda i: scores[i])
    next_params = candidates[best_idx]

    loss = cached_query(next_params)
    already_tried_keys.add(tuple(sorted(next_params.items(), key=lambda kv: kv[0])))
    X_train.append(encode(next_params))
    y_train.append(loss)

    print(f"BO iter {it+1}: tried {next_params} -> loss={loss:.6f} (pred mean={means[best_idx]:.4f}, std={stds[best_idx]:.4f})")

    if loss < bo_best_loss:
        bo_best_loss = loss
        bo_best_params = dict(next_params)
        print(f"  *** NEW BEST ***")

print()
print("Best from Bayesian optimization:", bo_best_params, "loss:", bo_best_loss)
print("Overall best (BO vs. everything so far):", min([(bo_best_params, bo_best_loss), (current, current_loss)], key=lambda x: x[1]))
print("FINAL total unique Oracle calls used:", len(cache))

BO iter 1: tried {'bootstrap': True, 'max_depth': None, 'max_features': 0.8, 'n_estimators': 12, 'min_samples_leaf': 8, 'min_samples_split': 20} -> loss=5.855447 (pred mean=0.6265, std=0.9947)
BO iter 2: tried {'bootstrap': False, 'max_depth': 3, 'max_features': 1, 'n_estimators': 3, 'min_samples_leaf': 8, 'min_samples_split': 2} -> loss=12.575675 (pred mean=0.3440, std=0.9997)
BO iter 3: tried {'bootstrap': True, 'max_depth': None, 'max_features': 0.4, 'n_estimators': 12, 'min_samples_leaf': 1, 'min_samples_split': 20} -> loss=7.139945 (pred mean=0.2703, std=0.9993)
BO iter 4: tried {'bootstrap': False, 'max_depth': 3, 'max_features': 1, 'n_estimators': 12, 'min_samples_leaf': 8, 'min_samples_split': 2} -> loss=12.575675 (pred mean=1.0431, std=0.9937)
BO iter 5: tried {'bootstrap': False, 'max_depth': None, 'max_features': 0.4, 'n_estimators': 3, 'min_samples_leaf': 1, 'min_samples_split': 20} -> loss=8.404048 (pred mean=0.4991, std=0.9982)
BO iter 6: tried {'bootstrap': True, 'max_de

In [19]:
N_MORE_BO_ITERS = 10

for it in range(N_MORE_BO_ITERS):
    unvisited = [combo_to_params(c) for c in all_combos_full
                 if tuple(sorted(combo_to_params(c).items(), key=lambda kv: kv[0])) not in already_tried_keys]
    candidates = random.sample(unvisited, min(150, len(unvisited)))
    X_cand = [encode(p) for p in candidates]

    means, stds = gp_predict(X_train, y_train, X_cand)
    scores = [m - KAPPA * s for m, s in zip(means, stds)]
    best_idx = min(range(len(candidates)), key=lambda i: scores[i])
    next_params = candidates[best_idx]

    loss = cached_query(next_params)
    already_tried_keys.add(tuple(sorted(next_params.items(), key=lambda kv: kv[0])))
    X_train.append(encode(next_params))
    y_train.append(loss)

    print(f"BO iter {it+1}: tried {next_params} -> loss={loss:.6f} (pred mean={means[best_idx]:.4f}, std={stds[best_idx]:.4f})")

    if loss < bo_best_loss:
        bo_best_loss = loss
        bo_best_params = dict(next_params)
        print(f"  *** NEW BEST ***")

print()
print("Best from Bayesian optimization (all iterations):", bo_best_params, "loss:", bo_best_loss)
print("Overall best (BO vs. everything so far):", min([(bo_best_params, bo_best_loss), (current, current_loss)], key=lambda x: x[1]))
print("FINAL total unique Oracle calls used:", len(cache))

BO iter 1: tried {'bootstrap': True, 'max_depth': 3, 'max_features': 1, 'n_estimators': 12, 'min_samples_leaf': 8, 'min_samples_split': 20} -> loss=8.962334 (pred mean=1.0566, std=0.9794)
BO iter 2: tried {'bootstrap': False, 'max_depth': 3, 'max_features': 0.4, 'n_estimators': 5, 'min_samples_leaf': 2, 'min_samples_split': 20} -> loss=10.653191 (pred mean=1.0562, std=0.8969)
BO iter 3: tried {'bootstrap': False, 'max_depth': None, 'max_features': 1, 'n_estimators': 12, 'min_samples_leaf': 8, 'min_samples_split': 5} -> loss=6.991275 (pred mean=1.7191, std=0.9857)
BO iter 4: tried {'bootstrap': False, 'max_depth': None, 'max_features': 0.8, 'n_estimators': 12, 'min_samples_leaf': 1, 'min_samples_split': 20} -> loss=6.076019 (pred mean=1.0168, std=0.9844)
BO iter 5: tried {'bootstrap': False, 'max_depth': 10, 'max_features': 1, 'n_estimators': 12, 'min_samples_leaf': 1, 'min_samples_split': 2} -> loss=10.168221 (pred mean=1.5847, std=0.9780)
BO iter 6: tried {'bootstrap': True, 'max_dept

## Optimization Approach & Validation

**1. Greedy coordinate descent with directional early stopping** — starting from a baseline 
   configuration, each hyperparameter was tuned via ordered-neighbor probing, moving in the 
   improving direction and stopping once it turned worse. Converged to loss = 4.260959 in 23 calls.

**2. Interaction grid checks** — 3x3 grids around the optimum for (max_depth x min_samples_leaf) 
   and (min_samples_split x min_samples_leaf) confirmed no nearby combination improved on it.

**3. Multi-restart validation** — greedy descent re-run from two starting points at opposite 
   corners of the search space; both converged to nearby but strictly worse losses (4.359, 
   4.295), confirming the original result sits in the best basin found.

**4. Random search** — 300 uniformly sampled, previously-untested configurations spanning the 
   full 3,072-point space; none improved on the result.

**5. Bayesian optimization** — a from-scratch Gaussian Process (squared-exponential kernel, 
   Cholesky-based regression, no external libraries) with UCB acquisition, run for 22 total 
   iterations over the remaining unvisited space; confirmed the same optimum.

**Final result:**
- Best hyperparameters: `{bootstrap: True, max_depth: 7, max_features: 0.6, n_estimators: 12, 
  min_samples_leaf: 2, min_samples_split: 5}`
- Best loss: **4.260959**
- Total Oracle calls used: **419** (~13.6% of the 30% allowance, ~13.6% of the full 3,072-point space)

In [20]:
print("=" * 50)
print("FINAL SUBMISSION SUMMARY")
print("=" * 50)
print()
print("Best hyperparameters:", current)
print("Best loss:", current_loss)
print("Total unique Oracle calls used:", len(cache))
print()
print("Full call log (all configs queried, in order):")
for i, (params, loss) in enumerate(call_log, 1):
    print(f"{i}. {params} -> loss={loss:.6f}")

FINAL SUBMISSION SUMMARY

Best hyperparameters: {'bootstrap': True, 'max_depth': 7, 'max_features': 0.6, 'n_estimators': 12, 'min_samples_leaf': 2, 'min_samples_split': 5}
Best loss: 4.26095923057704
Total unique Oracle calls used: 419

Full call log (all configs queried, in order):
1. {'bootstrap': True, 'max_depth': 5, 'max_features': 0.8, 'n_estimators': 8, 'min_samples_leaf': 2, 'min_samples_split': 5} -> loss=5.367585
2. {'bootstrap': True, 'max_depth': 7, 'max_features': 0.8, 'n_estimators': 8, 'min_samples_leaf': 2, 'min_samples_split': 5} -> loss=5.230342
3. {'bootstrap': True, 'max_depth': 10, 'max_features': 0.8, 'n_estimators': 8, 'min_samples_leaf': 2, 'min_samples_split': 5} -> loss=5.262685
4. {'bootstrap': True, 'max_depth': 7, 'max_features': 0.8, 'n_estimators': 5, 'min_samples_leaf': 2, 'min_samples_split': 5} -> loss=5.622181
5. {'bootstrap': True, 'max_depth': 7, 'max_features': 0.8, 'n_estimators': 12, 'min_samples_leaf': 2, 'min_samples_split': 5} -> loss=4.776581